In [ ]:
import sys
import os
import random
from pathlib import Path

import torch
import numpy as np
import pytorch_lightning as pl

In [ ]:
project_root = Path.cwd().parent 
sys.path.append(str(project_root))

In [ ]:
from img_generation.models.pixel_cnn import train_pixel_cnn
from img_generation.utils import show_imgs
from img_generation.datasets import Config, DatasetType, load_dataloaders, PIXEL_CNN_TRANSFORM

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # wymagane przez niektóre operacje CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
device

In [ ]:
config = Config(dataset_type=DatasetType.CIFAR10, data_dir="../data/cifar10", batch_size=128, num_workers=4)
dataloaders, classes = load_dataloaders(config=config, transforms=PIXEL_CNN_TRANSFORM)

print(f"All classes: {classes}")    # tiny-imagenet labels are of poor quality
print(f"Number of all classes: {len(classes)}")

In [ ]:
for x, y in dataloaders["train"]:
    print(f"Input img shape: {x.shape}")
    print(f"class: {y}")
    break

In [ ]:
for x, _ in dataloaders["train"]:
    input_channels, input_size = x.shape[1], x.shape[2] # (B, C, H, W)
    break

In [ ]:
channels_hidden = 64
model_path = "../pretrained_models/pixel_cnn"
epochs = 5

model, result = train_pixel_cnn(epochs=epochs, dataloaders=dataloaders, device=device, model_path=model_path, c_in=input_channels, c_hidden=channels_hidden)
test_res = result["test"][0]
print("Test bits per dimension: %4.3fbpd" % (test_res["test_loss"] if "test_loss" in test_res else test_res["test_bpd"]))

torch.save(model.state_dict(), f=os.path.join([model_path, "pixel_cnn.pth"]))

In [ ]:
num_samples = 16
pl.seed_everything(1)

samples = model.sample(img_shape=(num_samples, input_channels, input_size, input_size))
show_imgs(samples.cpu())